In [2]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/pima_diabetes_preprocessed.npz"
OUT_DIR = "./output_conditionaldiabetes"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42
N_BACKGROUND_SAMPLES = 100

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")
    print(f"Loading data from {npz_path}...")
    d = np.load(npz_path, allow_pickle=True)

    X_train, X_test = d["X_train"].astype(float), d["X_test"].astype(float)
    y_train, y_test = d["y_train"].astype(int).ravel(), d["y_test"].astype(int).ravel()
    feat_names = d["feature_names"].tolist() if "feature_names" in d else [f"f{i}" for i in range(X_train.shape[1])]
    return X_train, y_train, X_test, y_test, feat_names

X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
n_features = X_train.shape[1]
all_labels = sorted(np.unique(y_train).tolist())

trained_models = {}
for m_name in ["logistic", "svm", "random_forest"]:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

# Extract Background Dataset
bg_rng = np.random.default_rng(RANDOM_STATE)
bg_idx = bg_rng.choice(len(X_train), size=min(N_BACKGROUND_SAMPLES, len(X_train)), replace=False)
X_background = X_train[bg_idx]

@dataclass
class FeaturePlayers:
    indices: List[int]; data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class BackgroundExpectationUtility:
    def __init__(self, model, x_full, y_true, X_background, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.X_background = np.asarray(X_background, dtype=float)
        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.class_idx = self.all_labels.index(self.y_true)
        self.data = FeaturePlayers(indices=list(range(len(feature_names))), data_names=feature_names)

    def __call__(self, S):
        batch = self.X_background.copy()
        for col_idx in S:
            batch[:, int(col_idx)] = self.x_full[int(col_idx)]

        if hasattr(self.model, "predict_proba"):
            return float(np.mean(self.model.predict_proba(batch)[:, self.class_idx]))
        else:
            return float(np.mean(self.model.predict(batch) == self.y_true))

test_rng = np.random.default_rng(SAMPLE_SEED)
test_idx = test_rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value), ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus), ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index)
]

records = []
print(f"\nRunning explanation on {len(test_idx)} samples using {len(METHODS)} methods...")

for model_name, model in trained_models.items():
    for i, idx in enumerate(test_idx):
        util = BackgroundExpectationUtility(model, X_test[idx], y_test[idx], X_background, feature_names, all_labels)
        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = np.asarray(getattr(res, 'values', res), dtype=float).ravel()
                records.append({
                    "model": model_name, "test_index": int(idx), "y_true": int(y_test[idx]),
                    "method": method_name, "alloc_full": alloc.tolist(),
                    "alloc_named": {feature_names[j]: float(alloc[j]) for j in range(n_features)}
                })
            except Exception as e:
                print(f"Error in {method_name}: {e}")

df = pd.DataFrame(records)
out_file = os.path.join(OUT_DIR, "background_expectation_results.csv")
df.to_csv(out_file, index=False)
print(f"Done. Results saved to {out_file}")

Loading data from ../../datasets/pima_diabetes_preprocessed.npz...

Running explanation on 50 samples using 5 methods...
Done. Results saved to ./output_conditionaldiabetes/background_expectation_results.csv


In [3]:
import os
import sys
import warnings
import logging
import ast
from pathlib import Path
from dataclasses import dataclass
from typing import List, Iterator

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus
from sklearn.preprocessing import LabelEncoder

sys.path.append(os.path.abspath(".."))

from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index
from models.train_model import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_NPZ_PATH = "../../datasets/pima_diabetes_preprocessed.npz"
RESULTS_CSV_PATH = "./output_conditionaldiabetes/background_expectation_results.csv"
OUT_DIR = Path("./quantus_conditional_diabetes/quantus_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV_PATH = OUT_DIR / "quantus_results_full_extended.csv"

RANDOM_STATE = 42
N_BACKGROUND_SAMPLES = 100
# Providing 1-10 ensures it covers both Diabetes (8) and GMSC (10) automatically
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7, 8]

# Utilities
@dataclass
class FeaturePlayers:
    indices: List[int]; data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class BackgroundExpectationUtility:
    def __init__(self, model, x_full, y_true, X_background, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.X_background = np.asarray(X_background, dtype=float)
        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.class_idx = self.all_labels.index(self.y_true)
        self.data = FeaturePlayers(indices=list(range(len(feature_names))), data_names=feature_names)

    def __call__(self, S):
        batch = self.X_background.copy()
        for col_idx in S:
            batch[:, int(col_idx)] = self.x_full[int(col_idx)]

        if hasattr(self.model, "predict_proba"):
            return float(np.mean(self.model.predict_proba(batch)[:, self.class_idx]))
        else:
            return float(np.mean(self.model.predict(batch) == self.y_true))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        x_np = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else x
        if x_np.ndim == 3: x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1: x_np = x_np.reshape(1, -1)
        out = self.model.predict_proba(x_np) if self.is_clf else self.model.predict(x_np)
        if not self.is_clf and out.ndim == 1: out = out.reshape(-1, 1)
        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train, X_test = d["X_train"].astype(float), d["X_test"].astype(float)
    y_train, y_test = d["y_train"].astype(int).ravel(), d["y_test"].astype(int).ravel()
    feat_names = d["feature_names"].tolist() if "feature_names" in d else [f"f{i}" for i in range(X_train.shape[1])]
    return X_train, X_test, y_train, y_test, feat_names

# Metrics & perturbation
def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()
    F = arr.shape[1] if arr.ndim == 2 else arr.shape[2]

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs: target_slice[valid_idxs] = base_vec[valid_idxs]

    for b in range(arr.shape[0]):
        idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
        if arr.ndim == 2: apply_mask(arr_pert[b], idx_list)
        elif arr.ndim == 3: apply_mask(arr_pert[b, 0], idx_list)
    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    noise = np.random.uniform(0.01, 0.02, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b): return np.linalg.norm(np.array(a) - np.array(b))

def get_metrics(baseline_vec):
    kwargs = {"baseline": baseline_vec.reshape(1, 1, -1)}
    return {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(nr_runs=10, subset_size=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "selectivity": quantus.Selectivity(patch_size=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "monotonicity": quantus.Monotonicity(features_in_step=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "pixel_flipping": quantus.PixelFlipping(features_in_step=1, perturb_baseline="mean", perturb_func=perturb_tabular_robust, perturb_func_kwargs=kwargs, return_aggregate=True),
        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),
        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(nr_samples=5, perturb_std=0.1, norm_numerator=distance_euclidean, norm_denominator=distance_euclidean, perturb_func=perturb_uniform_robust, perturb_func_kwargs={}, return_aggregate=True),
        "max_sensitivity": quantus.MaxSensitivity(nr_samples=5, lower_bound=0.1, upper_bound=0.2, norm_numerator=quantus.norm_func.fro_norm, norm_denominator=quantus.norm_func.fro_norm, perturb_func=perturb_uniform_robust, perturb_func_kwargs={}, return_aggregate=True)
    }

def compute_aopc_manual(sk_model, X_test_2d, y_test_indices, attributions_2d, top_k_list, baseline):
    if not hasattr(sk_model, "predict_proba"): return np.nan

    base_probs = sk_model.predict_proba(X_test_2d)
    base_scores = base_probs[np.arange(len(y_test_indices)), y_test_indices]
    drops = []

    valid_k_list = [k for k in top_k_list if k <= X_test_2d.shape[1]]
    for k in valid_k_list:
        X_pert = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_idx = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_pert[i, top_idx] = baseline[top_idx]
        new_scores = sk_model.predict_proba(X_pert)[np.arange(len(y_test_indices)), y_test_indices]
        drops.append(np.mean(base_scores - new_scores))
    return np.mean(drops) if drops else np.nan

# Robust explainer wrapper for quantus
def make_explain_func(method_name, all_labels, X_background, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value, "least_core": exact_least_core,
        "nucleolus": exact_nucleolus, "gately": exact_gately_value,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        real_model = model.model if isinstance(model, SklearnTorchWrapper) else model
        attributions = []
        for i in range(len(inputs)):
            x = inputs[i].detach().cpu().numpy().flatten() if hasattr(inputs[i], "detach") else inputs[i].flatten()
            y = int(targets[i].item()) if hasattr(targets[i], "item") else int(targets[i])
            real_label = real_model.classes_[y] if hasattr(real_model, "classes_") else y

            util = BackgroundExpectationUtility(real_model, x, real_label, X_background, feature_names, all_labels)
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception:
                attributions.append(np.zeros((1, len(x))))
        return np.array(attributions)
    return explain_func

# Main
def main():
    if not os.path.exists(RESULTS_CSV_PATH):
        print(f"ERROR: {RESULTS_CSV_PATH} not found. Run Explainer script first!")
        return

    X_train, X_test, y_train, y_test, feature_names = load_data_strict()
    baseline = np.mean(X_train, axis=0)
    all_labels = sorted(np.unique(y_train).tolist())

    le = LabelEncoder()
    le.fit(y_train)

    rng = np.random.default_rng(RANDOM_STATE)
    bg_idx = rng.choice(len(X_train), size=min(N_BACKGROUND_SAMPLES, len(X_train)), replace=False)
    X_background = X_train[bg_idx]

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset_indices = le.transform(y_test[valid_indices])

    x_batch = X_subset[:, np.newaxis, :]

    cols = ["model", "method", "faithfulness_corr", "faithfulness_estimate", "selectivity", "monotonicity", "pixel_flipping", "sparseness", "complexity", "local_lipschitz_estimate", "max_sensitivity", "aopc_manual"]
    if not OUT_CSV_PATH.exists():
        pd.DataFrame(columns=cols).to_csv(OUT_CSV_PATH, index=False)
        processed_pairs = set()
    else:
        existing_df = pd.read_csv(OUT_CSV_PATH)
        processed_pairs = set(zip(existing_df["model"], existing_df["method"]))
        print(f"Resuming... Found {len(processed_pairs)} already processed entries.")

    metrics = get_metrics(baseline)

    for model_name in df_results["model"].unique():
        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()
        df_model = df_results[df_results["model"] == model_name]

        for method_name in df_results["method"].unique():
            if (model_name, method_name) in processed_pairs:
                print(f"Skipping {model_name} - {method_name} (Already in CSV)")
                continue

            print(f"Evaluating {model_name} - {method_name}...")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if not row.empty:
                    val_str = row.iloc[0]["alloc_full"]
                    val_arr = np.array(ast.literal_eval(val_str)) if isinstance(val_str, str) else np.array(val_str)
                    attributions_list.append(val_arr)
                else:
                    attributions_list.append(np.zeros(X_train.shape[1]))

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}
            y_faith = y_subset_indices if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset_indices)

            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try: row_stats[m_name] = np.nanmean(metric_fn(model=torch_model, x_batch=x_batch, y_batch=y_faith, a_batch=a_batch_3d, channel_first=True, softmax=False))
                except Exception: row_stats[m_name] = np.nan

            try: row_stats["aopc_manual"] = compute_aopc_manual(sk_model, X_subset, y_subset_indices, a_batch_2d, AOPC_TOPK_LIST, baseline)
            except Exception: row_stats["aopc_manual"] = np.nan

            explain_func = make_explain_func(method_name, all_labels, X_background, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                if model_name == "random_forest":
                     row_stats[m_name] = np.nan
                     continue
                try: row_stats[m_name] = np.nanmean(metrics[m_name](model=torch_model, x_batch=x_batch, y_batch=y_subset_indices, explain_func=explain_func, channel_first=True, softmax=False))
                except Exception: row_stats[m_name] = np.nan

            df_row = pd.DataFrame([row_stats]).reindex(columns=cols)
            df_row.to_csv(OUT_CSV_PATH, mode='a', header=False, index=False)
            print(f"  -> Saved results for {model_name} - {method_name}")

    print("\nFinished evaluating all models and methods!")

if __name__ == "__main__":
    main()

Evaluating logistic - shapley...
  -> Saved results for logistic - shapley
Evaluating logistic - least_core...
  -> Saved results for logistic - least_core
Evaluating logistic - nucleolus...
  -> Saved results for logistic - nucleolus
Evaluating logistic - gately...
  -> Saved results for logistic - gately
Evaluating logistic - banzhaf...
  -> Saved results for logistic - banzhaf
Evaluating svm - shapley...
  -> Saved results for svm - shapley
Evaluating svm - least_core...
  -> Saved results for svm - least_core
Evaluating svm - nucleolus...
  -> Saved results for svm - nucleolus
Evaluating svm - gately...
  -> Saved results for svm - gately
Evaluating svm - banzhaf...
  -> Saved results for svm - banzhaf
Evaluating random_forest - shapley...
  -> Saved results for random_forest - shapley
Evaluating random_forest - least_core...
  -> Saved results for random_forest - least_core
Evaluating random_forest - nucleolus...
  -> Saved results for random_forest - nucleolus
Evaluating random_f